 # Stochastic Methods in Finance

## Importing packages

In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

## Importing and inspecting the dataset

In [49]:
data = pd.read_csv("/Users/ludozzi/Visual Studio Code/Stochastic-Methods-in-Finance---FS-25/data/Microsoft_2015_2025.csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2515 entries, 0 to 2514
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Date        2515 non-null   object
 1   Close/Last  2515 non-null   object
 2   Volume      2515 non-null   int64 
 3   Open        2515 non-null   object
 4   High        2515 non-null   object
 5   Low         2515 non-null   object
dtypes: int64(1), object(5)
memory usage: 118.0+ KB


In [50]:
data.head()

,Date,Close/Last,Volume,Open,High,Low
0,04/28/2025,$391.16,16579430,$391.955,$392.74,$386.638
1,04/25/2025,$391.85,18973170,$387.00,$392.16,$384.60
2,04/24/2025,$387.30,22232290,$375.695,$388.45,$375.19
3,04/23/2025,$374.39,20545530,$376.06,$380.39,$373.02
4,04/22/2025,$366.82,19484990,$363.375,$367.77,$359.8602


In [51]:
data.tail()

,Date,Close/Last,Volume,Open,High,Low
2510,05/05/2015,$47.60,50364250,$47.82,$48.16,$47.31
2511,05/04/2015,$48.24,33986240,$48.37,$48.87,$48.18
2512,05/01/2015,$48.655,36432670,$48.58,$48.875,$48.40
2513,04/30/2015,$48.64,64127280,$48.70,$49.54,$48.60
2514,04/29/2015,$49.06,47665440,$48.72,$49.31,$48.50


## Preprocessing

In [52]:
# Converting the date column to datetime
data['Date'] = pd.to_datetime(data['Date'])
print(data["Date"].dtype)

# Setting the date as the index
data.set_index('Date', inplace=True)

# Setting the variables as numeric
# Remove dollar signs and any other non-numeric characters, then convert to numeric
data['Close/Last'] = data['Close/Last'].replace({'\$': '', ',': ''}, regex=True).astype(float)
data['Open'] = data['Open'].replace({'\$': '', ',': ''}, regex=True).astype(float)
data['High'] = data['High'].replace({'\$': '', ',': ''}, regex=True).astype(float)
data['Low'] = data['Low'].replace({'\$': '', ',': ''}, regex=True).astype(float)
data['Volume'] = data['Volume'].replace({',': ''}, regex=True).astype(int)

# Check the types of the columns
print(data.dtypes)

datetime64[ns]
Close/Last    float64
Volume          int64
Open          float64
High          float64
Low           float64
dtype: object


<>:10: SyntaxWarning: invalid escape sequence '\$'
<>:11: SyntaxWarning: invalid escape sequence '\$'
<>:12: SyntaxWarning: invalid escape sequence '\$'
<>:13: SyntaxWarning: invalid escape sequence '\$'
<>:10: SyntaxWarning: invalid escape sequence '\$'
<>:11: SyntaxWarning: invalid escape sequence '\$'
<>:12: SyntaxWarning: invalid escape sequence '\$'
<>:13: SyntaxWarning: invalid escape sequence '\$'
/var/folders/wn/nf_gxvtd0zsfbsgzc8y_qpn40000gn/T/ipykernel_72046/126352498.py:10: SyntaxWarning: invalid escape sequence '\$'
  data['Close/Last'] = data['Close/Last'].replace({'\$': '', ',': ''}, regex=True).astype(float)
/var/folders/wn/nf_gxvtd0zsfbsgzc8y_qpn40000gn/T/ipykernel_72046/126352498.py:11: SyntaxWarning: invalid escape sequence '\$'
  data['Open'] = data['Open'].replace({'\$': '', ',': ''}, regex=True).astype(float)
/var/folders/wn/nf_gxvtd0zsfbsgzc8y_qpn40000gn/T/ipykernel_72046/126352498.py:12: SyntaxWarning: invalid escape sequence '\$'
  data['High'] = data['High'].re

In [53]:
data.head()

,Close/Last,Volume,Open,High,Low
Date,,,,,
2025-04-28,391.16,16579430,391.955,392.74,386.6380
2025-04-25,391.85,18973170,387.000,392.16,384.6000
2025-04-24,387.30,22232290,375.695,388.45,375.1900
2025-04-23,374.39,20545530,376.060,380.39,373.0200
2025-04-22,366.82,19484990,363.375,367.77,359.8602


## Tasks

### 1. Creating the binomial tree

In [54]:

# Calculate daily returns and the standard deviation (volatility)
data['Return'] = data['Close/Last'].pct_change()
sigma_daily = data['Return'].std()  # Sample standard deviation of daily returns
sigma_annual = sigma_daily * np.sqrt(250)  # Annualize the volatility (assuming 250 trading days)

# Initial stock price S0 (price on April 28, 2025)
S_0 = data.loc['2025-04-28', 'Close/Last']  # Update with the correct date or your preferred method

# Given parameters
n = 25  # Number of periods
r = 0.01  # Risk-free rate (1% per annum)
T = 25  # Assuming 1 year for the option maturity

# Calculate the up and down factors
dt = T / n  # Time step (recalling, daily data)
u = np.exp(sigma_annual * np.sqrt(dt))  # Up factor
d = np.exp(-sigma_annual * np.sqrt(dt))  # Down factor

# Risk-neutral probability
p = 0.5  # Risk-neutral probability (as per your assignment)

# Step 1: Create the binomial tree for St
# Initialize an empty array for the binomial tree (prices at each node)
tree = np.zeros((n + 1, n + 1))

# Set the initial price at the root (S_0)
tree[0, 0] = S_0

# Fill in the tree with prices for each node
for i in range(1, n + 1):
    for j in range(i + 1):
        tree[i, j] = S_0 * (u ** j) * (d ** (i - j))

# Print the binomial tree for visual inspection
print("Binomial Tree for Stock Prices (S_t):")
print(tree)


Binomial Tree for Stock Prices (S_t):
[[3.91160000e+02 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [2.98421247e+02 5.12718672e+02 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [2.27669600e+02 3.91160000e+02 6.72053474e+02 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e

### 2. Average prices for each node

In [55]:
# Initialize a list to store the average prices
average_prices = np.zeros_like(tree)

# Computing the average price for each node
for i in range(n + 1):
    for j in range(i + 1):
        # Average price for node (i, j) is the mean of all prices along the path
        average_prices[i, j] = np.mean(tree[i, :j+1])

# Print the average prices for visual inspection
print("Average Prices for Each Node:")
print(average_prices)


Average Prices for Each Node:
[[3.91160000e+02 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [2.98421247e+02 4.05569960e+02 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [2.27669600e+02 3.09414800e+02 4.30294358e+02 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00

### 3. Payoffs of the Asian option
##### Since we do not have a value for the strike price, we assume that the European Asian call option is at the money, i.e. the strike price is equal the initial stock price.

In [56]:
# Assuming the call is ATM
K = S_0

# Calculate the payoff at each terminal node (t = n)
payoff = np.maximum(average_prices[n, :] - K, 0)  # Payoff for Asian call option

# Print the payoff at each terminal node
print("Payoff at each terminal node:")
print(payoff)


Payoff at each terminal node:
[    0.             0.             0.             0.
     0.             0.             0.             0.
     0.             0.             0.             0.
     0.             0.             0.             0.
     0.           202.6409129    575.37998596  1186.44775602
  2190.28419192  3842.45002483  6566.38963195 11064.57643386
 18503.71634526 30823.62510267]


### 4. Risk-neutral probabilities

In [57]:
p_risk_neutral = (np.exp(r * dt) - d) / (u - d)
q_risk_netural = 1 - p_risk_neutral

### 5. Backward induction

In [58]:
# Since the compunding risk-free rate per annum is given, we have to convert it to a daiily rate 
r_daily = (1 + r) ** (1 / 250) - 1

# Initialize the option values at the terminal nodes (n = 25)
option_values = np.zeros_like(average_prices)

# Calculate the payoff at the terminal nodes (at t = n)
# The payoff is max(average_price - K, 0) for each terminal node
payoff_terminal = np.maximum(average_prices[n, :] - K, 0)

# Set the option values at the terminal nodes
option_values[n, :] = payoff_terminal

# Applying backward induction to calculate the option price at t = 0
for i in range(n - 1, -1, -1):  # Iterate backward from t = n-1 to t = 0
    for j in range(i + 1):  # There are i+1 nodes at each time step
        # Option value at each node is the discounted expected value of future values
        option_values[i, j] = np.exp(-r_daily * dt) * (p_risk_neutral * option_values[i + 1, j + 1] + q_risk_netural * option_values[i + 1, j])

# The option price at the root node (t = 0) is the price of the Asian option
asian_option_price = option_values[0, 0]

# Print the price of the Asian option at t = 0
print(f"Arbitrage-free price of the Asian option at t = 0: {asian_option_price}")


Arbitrage-free price of the Asian option at t = 0: 7.446218744563005


### 6. Robustness check
#### To analyse the sensitivity of the option, we can change the risk-free rate and the volatility to understand how the price of the option changes

In [59]:
# Assuming hypothetical interest rates and volatilities for the analysis
for r in interest_rates:
    for sigma in volatilities:
        # Calculate option price for each combination of interest rate and volatility
        # Placeholder option pricing formula, replace with actual calculation
        option_price = S_0 * np.exp(-r * 1) - K  # Simple formula for demonstration
        
        # Store the result in the dictionary with the key as (interest rate, volatility)
        results[(r, sigma)] = option_price
        print(f"Interest rate: {r*100}%, Volatility: {sigma*100}%, Option Price: {option_price}")

Interest rate: 0.5%, Volatility: 10.0%, Option Price: -1.9509186389904016
Interest rate: 0.5%, Volatility: 20.0%, Option Price: -1.9509186389904016
Interest rate: 0.5%, Volatility: 30.0%, Option Price: -1.9509186389904016
Interest rate: 1.0%, Volatility: 10.0%, Option Price: -3.8921070306753904
Interest rate: 1.0%, Volatility: 20.0%, Option Price: -3.8921070306753904
Interest rate: 1.0%, Volatility: 30.0%, Option Price: -3.8921070306753904
Interest rate: 2.0%, Volatility: 10.0%, Option Price: -7.745486949329631
Interest rate: 2.0%, Volatility: 20.0%, Option Price: -7.745486949329631
Interest rate: 2.0%, Volatility: 30.0%, Option Price: -7.745486949329631


### 7. Price approximation
#### Here, we use the normal approximation of the binomial distribution to compute the price of the Asian call option

In [60]:
# Mean of the average price
mean_avg = S_0 * np.exp(r_daily * T)

# Daily variances of the average price
sigma_avg_squared = (sigma_daily ** 2 / n) * (1 - np.exp(-2 * r_daily * T)) / (2 * r_daily)
sigma_avg = np.sqrt(sigma_avg_squared)

# Cumulative distribution functions of the normal distribution
d1 = (np.log(mean_avg / K) + 0.5 * sigma_avg_squared) / sigma_avg
d2 = d1 - sigma_avg

price_normal_approx = np.exp(-r_daily * T) * (S_0 * norm.cdf(d1) - K * norm.cdf(d2))
print(f"Asian option price using normal approximation: {price_normal_approx}")

Asian option price using normal approximation: 2.6622640868590453
